[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balloontip/deep-learning/blob/main/chapter-09/09-08-Variational-Autoencoder-VAE-Synthetic-Data-Generation.ipynb)

End-to-End Example 1: Training a Variational Autoencoder (VAE) for Synthetic Data Generation. Build and train a Variational Autoencoder (VAE) in PyTorch to learn a compact latent representation of synthetic sports-like vectors and generate new synthetic samples through latent space sampling. This project demonstrates the encoder-decoder architecture, the reparameterization trick, reconstruction and KL-divergence losses, and end-to-end VAE training.

In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# -------------------------------------------------------
# 1. Set Device and Prepare Synthetic Data
# -------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

num_samples = 3000
data_dim = 100      # Think of this as a flattened 10 × 10 image vector
latent_dim = 2      # Compress each sample into 2 latent numbers

np.random.seed(42)
torch.manual_seed(42)

# Three synthetic clusters representing three sports-like object groups
cluster1 = np.random.normal(loc=0.2, scale=0.1, size=(num_samples // 3, data_dim))
cluster2 = np.random.normal(loc=0.5, scale=0.1, size=(num_samples // 3, data_dim))
cluster3 = np.random.normal(loc=0.8, scale=0.1, size=(num_samples // 3, data_dim))

X = np.concatenate([cluster1, cluster2, cluster3], axis=0)

# BCE loss expects target values between 0 and 1
X = np.clip(X, 0, 1)

X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

dataset = TensorDataset(X_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# -------------------------------------------------------
# 2. Define the VAE Model
# -------------------------------------------------------
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 50),
            nn.ReLU(),
            nn.Linear(50, 2 * latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 50),
            nn.ReLU(),
            nn.Linear(50, input_dim),
            nn.Sigmoid()
        )

        self.latent_dim = latent_dim

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        z = mu + std * epsilon
        return z

    def forward(self, x):
        encoded = self.encoder(x)

        mu = encoded[:, :self.latent_dim]
        logvar = encoded[:, self.latent_dim:]

        z = self.reparameterize(mu, logvar)
        reconstructed_x = self.decoder(z)

        return reconstructed_x, mu, logvar

# -------------------------------------------------------
# 3. Define the Loss Function and Optimizer
# -------------------------------------------------------
def vae_loss(reconstructed_x, x, mu, logvar):
    reconstruction_loss = nn.functional.binary_cross_entropy(
        reconstructed_x,
        x,
        reduction="sum"
    )

    kl_divergence = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    return reconstruction_loss + kl_divergence

model = VAE(input_dim=data_dim, latent_dim=latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------------------------------------------
# 4. Train the VAE
# -------------------------------------------------------
num_epochs = 100

for epoch in range(num_epochs):
    total_loss = 0.0

    for batch in dataloader:
        vectors = batch[0]

        optimizer.zero_grad()

        reconstructed_vectors, mu, logvar = model(vectors)
        loss = vae_loss(reconstructed_vectors, vectors, mu, logvar)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        average_loss = total_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch + 1}/{num_epochs}], Average Loss: {average_loss:.4f}")

print("Training finished!")

# -------------------------------------------------------
# 5. Generate a New Synthetic Example
# -------------------------------------------------------
print("\nGenerating a new synthetic sports-like vector:")

with torch.no_grad():
    random_latent_vector = torch.randn(1, latent_dim).to(device)
    new_vector = model.decoder(random_latent_vector)

    print(f"Generated vector shape: {new_vector.shape}")
    print("First 10 generated values:")
    print(new_vector.cpu().numpy().flatten()[:10])


Using device: cpu
Epoch [10/100], Average Loss: 58.8322
Epoch [20/100], Average Loss: 58.6387
Epoch [30/100], Average Loss: 58.5253
Epoch [40/100], Average Loss: 58.4669
Epoch [50/100], Average Loss: 58.4354
Epoch [60/100], Average Loss: 58.4421
Epoch [70/100], Average Loss: 58.3941
Epoch [80/100], Average Loss: 58.3905
Epoch [90/100], Average Loss: 58.3925
Epoch [100/100], Average Loss: 58.3255
Training finished!

Generating a new synthetic sports-like vector:
Generated vector shape: torch.Size([1, 100])
First 10 generated values:
[0.64336926 0.63985854 0.630297   0.64072055 0.65019983 0.642337
 0.64809364 0.6381939  0.64340985 0.64563406]
